# SpectraCore · CAMSPECTOR — make all Track 4 outputs on free Google Colab

**Nothing runs on your laptop and nothing big is stored.** EPIC-KITCHENS videos are downloaded
one at a time, processed and deleted. Results are saved to your Google Drive as they finish, so if
Colab disconnects, open this notebook again and press **Runtime → Run all**: it continues where it
stopped.

**Before you start**
1. Upload `SpectraCore_Track4_Submission.zip` to the top level of **My Drive**.
2. **Runtime → Change runtime type → T4 GPU** (a CPU runtime also works, just slower).
3. **Runtime → Run all**, and allow Google Drive access when asked.

Expect a few hours in total, mostly downloading ~184 GB of EPIC video (never stored at once).
At the end, `SpectraCore_upload.zip` appears in My Drive: that is the folder to submit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE = '/content/drive/MyDrive'
ZIP = f'{DRIVE}/SpectraCore_Track4_Submission.zip'
OUT = f'{DRIVE}/SpectraCore_outputs'            # the JSONs (kept between sessions)
WEIGHTS = f'{DRIVE}/SpectraCore_weights'        # CLIP + verb prior (~600 MB, downloaded once)
FPV = f'{DRIVE}/SpectraCore_fpvsum_videos'      # FPVSum videos (small)
RUNLOG = f'{DRIVE}/SpectraCore_run_log.csv'
for d in (OUT, WEIGHTS, FPV):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(ZIP), 'Upload SpectraCore_Track4_Submission.zip to My Drive first'
!nvidia-smi -L || echo "No GPU: running on CPU (slower, still works)"

In [ ]:
!rm -rf /content/sub && mkdir -p /content/sub && unzip -q "{ZIP}" -d /content/sub
%cd /content/sub/SpectraCore_Track4_Submission/SpectraCore
!pip -q install av transformers yt-dlp
!rm -rf weights && ln -s "{WEIGHTS}" weights
!python code/scripts/download_weights.py

Quick self-test of the output format, validator and model heads (about 10 seconds).

In [ ]:
!PYTHONPATH=code python -m pytest code/tests/test_contract.py code/tests/test_validate.py code/tests/test_zeroshot.py -q

## FPVSum videos (official crawler, with yt-dlp instead of the retired youtube-dl)

If YouTube blocks Colab ("Sign in to confirm you're not a bot"), run the same crawler on any
other computer, then upload the videos into **My Drive/SpectraCore_fpvsum_videos**, each named
`<video_id>.mp4` as in `manifest.csv`. The two Scuba videos that are gone from YouTube are excluded.

In [ ]:
have = [p for p in glob.glob(f'{FPV}/**/*', recursive=True) if p.lower().endswith(('.mp4', '.mkv', '.webm'))]
print(len(have), 'FPVSum video files already in Drive')
if len(have) < 47:
    !git clone -q --depth 1 https://github.com/azuxmioy/fpvsum /content/fpvsum || true
    !ln -sf "$(which yt-dlp)" /usr/local/bin/youtube-dl
    !cd /content/fpvsum/crawler && chmod +x fetch_fpvsum_videos.sh && ./fetch_fpvsum_videos.sh "{FPV}" FPVSum_videolist.csv
    !ls "{FPV}" | wc -l

## Run every video (resumable)

Watch for `model=True` in the log lines. `model=False` means that video fell back to the simple
baseline; the summary at the end counts these.

In [ ]:
!python code/scripts/stream_run.py --out "{OUT}" --fpvsum-dir "{FPV}" --work /content/epic_tmp --log "{RUNLOG}"

## Validate and build the upload package

In [ ]:
!python validate.py --dir "{OUT}"
!rm -rf outputs && cp -r "{OUT}" outputs && cp "{RUNLOG}" experiments/run_log.csv
!cd .. && rm -f "{DRIVE}/SpectraCore_upload.zip" && zip -qr "{DRIVE}/SpectraCore_upload.zip" SpectraCore -x "SpectraCore/data/*" "SpectraCore/.cache/*"
print('Upload package ready in My Drive: SpectraCore_upload.zip')